# aggregate center analysis

aggregate center (AC) of the VTAs of the best ΔLEDD responders in a training set, and whether proximity to the AC predicts ΔLEDD in a held-out test set. UCSF STN aggregate centers are then compared with published PD sweet spots.

**note:** the aggregate centers and test-set metrics (sections 4 and 6-8) are computed from patient-level VTA masks (NIfTI) exported from Lead-DBS, and everything after section 8 builds on them. these images are not shared in this repository, so the notebook will not run in full without them. researchers interested in the imaging data can contact the corresponding author about collaboration.

## 0. setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy.ndimage import center_of_mass, label as cc_label
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy import stats

from sklearn.model_selection import GroupShuffleSplit
from sklearn.cluster import KMeans

import statsmodels.api as sm

warnings.filterwarnings("ignore")

## 1. configuration

In [ ]:
# paths
XLSX_PATH = "<path to outcomes_data.xlsx>"
OUT_DIR   = "figures_aggregate_center"
os.makedirs(OUT_DIR, exist_ok=True)

# folder with one sub-folder per patient containing Lead-DBS outputs
VTA_ROOT  = "<path to Lead-DBS patient folders>"

# VTA filename, relative to VTA_ROOT/<patient_id>/
VTA_SUBPATH = (
    "derivatives/leaddbs/sub-leads/stimulations/MNI152NLin2009bAsym/stim/"
    "resamp_sub-leads_sim-binary_model-{model}_hemi-{hemi}.nii"
)
VTA_TEMPLATES = [
    VTA_SUBPATH.format(model="simbio",    hemi="{hemi}"),
    VTA_SUBPATH.format(model="simbio",    hemi="{hemi}") + ".gz",
    VTA_SUBPATH.format(model="fastfield", hemi="{hemi}"),
    VTA_SUBPATH.format(model="fastfield", hemi="{hemi}") + ".gz",
]

# column names in outcomes_data.xlsx
COL_ID         = "id"
COL_TARGET     = "Target"       # STN / GPi
COL_LATERALITY = "Laterality"   # Bi / L / R
COL_DELTA      = "delta_LEDD"   # positive = more medication reduction
COL_AGE        = "Age"
COL_SEX        = "Sex"

GROUPS = ["STN_L", "STN_R", "GPI_L", "GPI_R"]

# analysis parameters (KMEANS_K is set in section 5)
RANDOM_STATE       = 7
TEST_SIZE          = 0.30
KMEANS_N_INIT      = 50
MIN_TRAIN_GROUP    = 8
MIN_TRAIN_BEST_BIN = 4
MIN_TEST_REG       = 5

SPHERE_RADIUS_MM   = 2.0
ROBUST_NORM        = "HuberT"
N_PERMUTATIONS     = 5000

# figure style
STN_PALETTE = ["#f8bbd0", "#f06292", "#ad1457"]
GPI_PALETTE = ["#ffe0b2", "#ffb74d", "#f57c00"]
ALPHA_HIST  = 0.65
FIG_DPI     = 300
FIG_EXT     = "tiff"

plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})



## 2. data loading and cleaning

In [ ]:
df_raw = pd.read_excel(XLSX_PATH)

# normalise strings
df_raw[COL_ID]         = df_raw[COL_ID].astype(str).str.strip()
df_raw[COL_TARGET]     = df_raw[COL_TARGET].astype(str).str.strip().str.upper()
df_raw[COL_LATERALITY] = df_raw[COL_LATERALITY].astype(str).str.strip().str.upper()

# numeric coercion
df_raw[COL_DELTA] = pd.to_numeric(df_raw[COL_DELTA], errors="coerce")
df_raw[COL_AGE]   = pd.to_numeric(df_raw[COL_AGE],   errors="coerce")
df_raw["Sex_binary"] = df_raw[COL_SEX].astype(str).str.upper().map({"M": 1, "F": 0})

# one row per hemisphere: a bilateral patient gives an L and an R row.
# the train/test split is done on id so both hemispheres stay in the same set.
rows = []
for _, r in df_raw.iterrows():
    target = r[COL_TARGET]
    lat    = r[COL_LATERALITY]

    if lat == "BI":
        sides = ["L", "R"]
    elif lat in ("L", "R"):
        sides = [lat]
    else:
        print(f"  unknown laterality '{lat}' for {r[COL_ID]}, skipping")
        continue

    for side in sides:
        rows.append({
            COL_ID:        r[COL_ID],
            "Group":       f"{target}_{side}",
            "Side":        side,
            COL_TARGET:    target,
            COL_LATERALITY: lat,
            COL_DELTA:     r[COL_DELTA],
            COL_AGE:       r[COL_AGE],
            "Sex_binary":  r["Sex_binary"],
        })

df = pd.DataFrame(rows)

# drop rows with missing outcome or covariates
df = df[df["Group"].isin(GROUPS)].copy()
df = df.dropna(subset=[COL_ID, "Group", COL_DELTA, COL_AGE, "Sex_binary"]).reset_index(drop=True)

print(f"Raw rows in Excel   : {df_raw.shape[0]}")
print(f"Expanded (per hemi) : {df.shape[0]}")
print("  (bilateral patients each appear twice)")
print("\nGroup counts:")
print(df["Group"].value_counts().to_string())
print("\nLaterality breakdown within expanded df:")
print(df.groupby(["Group", COL_LATERALITY]).size().to_string())
print(f"\ndelta_LEDD NaN: {df[COL_DELTA].isna().sum()}")

In [ ]:
# cohort summary
print("Patient cohort summary")

summary = df_raw.groupby([COL_TARGET, COL_LATERALITY]).agg(
    n_patients=(COL_ID, "count"),
    n_missing_LEDD=(COL_DELTA, lambda x: x.isna().sum()),
    mean_LEDD=(COL_DELTA, "mean"),
    std_LEDD=(COL_DELTA, "std"),
    mean_age=(COL_AGE, "mean"),
    n_female=(COL_SEX, lambda x: (x.str.upper() == "F").sum()),
).round(2)

display(summary)

print(f"\nTotal unique patients  : {df_raw[COL_ID].nunique()}")
print(f"  STN                  : {(df_raw[COL_TARGET]=='STN').sum()}")
print(f"  GPi                  : {(df_raw[COL_TARGET]=='GPI').sum()}")
print(f"  Bilateral (Bi)       : {(df_raw[COL_LATERALITY]=='BI').sum()}")
print(f"  Unilateral Left (L)  : {(df_raw[COL_LATERALITY]=='L').sum()}")
print(f"  Unilateral Right (R) : {(df_raw[COL_LATERALITY]=='R').sum()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, tgt in zip(axes, ["STN", "GPI"]):
    sub = df_raw[df_raw[COL_TARGET] == tgt][COL_DELTA].dropna()
    ax.hist(sub, bins=20, color="#f06292" if tgt == "STN" else "#ffb74d",
            edgecolor="white", linewidth=0.5)
    ax.axvline(sub.mean(), color="black", lw=1.5, linestyle="--", label=f"mean={sub.mean():.1f}")
    ax.set_title(f"{tgt}  (n={len(sub)})", fontsize=12)
    ax.set_xlabel("delta_LEDD"); ax.set_ylabel("N patients")
    ax.legend(fontsize=9)
plt.suptitle("ΔLEDD distribution by target (all patients)", fontsize=13, fontweight="bold")
plt.show()

## 3. subject-level train / test split

In [ ]:
unique_ids = df[COL_ID].unique()
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
idx = np.arange(len(unique_ids))
train_idx, test_idx = next(gss.split(idx, groups=unique_ids))

train_ids = set(unique_ids[train_idx])
test_ids  = set(unique_ids[test_idx])

print(f"Unique patients : {len(unique_ids)}")
print(f"Train           : {len(train_ids)}")
print(f"Test (held-out) : {len(test_ids)}")
print(f"Overlap         : {len(train_ids & test_ids)}  (must be 0)")

## 4. VTA loading and spatial helpers (requires NIfTI)

In [ ]:
def subject_mask_path(patient_id: str, hemi: str) -> str | None:
    # tries simbio and fastfield models, with and without .gz
    for tmpl in VTA_TEMPLATES:
        p = os.path.join(VTA_ROOT, patient_id, tmpl.format(hemi=hemi))
        if os.path.exists(p):
            return p
    return None


def load_mask(patient_id: str, hemi: str) -> nib.Nifti1Image | None:
    p = subject_mask_path(patient_id, hemi)
    return nib.load(p) if p else None


def assert_same_grid(imgs: list) -> None:
    ref = imgs[0]
    for i, im in enumerate(imgs[1:], 1):
        if im.shape != ref.shape:
            raise ValueError(f"Shape mismatch at idx {i}: {im.shape} vs {ref.shape}")
        if not np.allclose(im.affine, ref.affine, atol=1e-4):
            raise ValueError(f"Affine mismatch at idx {i}")


def largest_cc(vol: np.ndarray) -> np.ndarray:
    # keep largest connected component
    b = (vol > 0).astype(np.uint8)
    lab, n = cc_label(b)
    if n == 0:
        return b
    counts = np.bincount(lab.ravel())
    counts[0] = 0
    return (lab == counts.argmax()).astype(np.uint8)


def ijk_to_xyz(ijk: np.ndarray, affine: np.ndarray) -> np.ndarray:
    ijk_h = np.c_[ijk, np.ones(len(ijk))]
    return (affine @ ijk_h.T).T[:, :3]


def xyz_to_ijk(xyz: np.ndarray, affine: np.ndarray) -> np.ndarray:
    inv = np.linalg.inv(affine)
    xyz_h = np.c_[xyz, np.ones(len(xyz))]
    return (inv @ xyz_h.T).T[:, :3]


def voxel_sizes_mm(affine: np.ndarray) -> np.ndarray:
    return np.sqrt((affine[:3, :3] ** 2).sum(axis=0))


def compute_ac_xyz(vols: list, affine: np.ndarray) -> np.ndarray:
    # weighted center of mass of summed VTAs (MNI mm)
    summed = np.sum(vols, axis=0).astype(float)
    if summed.max() == 0:
        return np.full(3, np.nan)
    com_ijk = np.array(center_of_mass(summed))
    return ijk_to_xyz(com_ijk[None, :], affine)[0]


def nearest_voxel_distance(vol: np.ndarray, affine: np.ndarray, center: np.ndarray) -> float:
    # distance (mm) from AC to nearest VTA voxel
    pts = np.argwhere(vol > 0)
    if pts.size == 0 or np.any(np.isnan(center)):
        return np.nan
    d, _ = cKDTree(ijk_to_xyz(pts, affine)).query(center, k=1)
    return float(d)


def sphere_overlap(vol: np.ndarray, affine: np.ndarray, center: np.ndarray,
                   radius_mm: float = SPHERE_RADIUS_MM) -> float:
    # fraction of voxels in a radius_mm sphere around the AC that are in the VTA
    if np.any(np.isnan(center)):
        return np.nan
    vs     = voxel_sizes_mm(affine)
    rad_v  = np.ceil(radius_mm / vs).astype(int)
    c_ijk  = np.round(xyz_to_ijk(center[None, :], affine)[0]).astype(int)
    lo     = np.maximum(c_ijk - rad_v, 0)
    hi     = np.minimum(c_ijk + rad_v + 1, np.array(vol.shape))
    grid   = np.array(np.meshgrid(*[np.arange(lo[i], hi[i]) for i in range(3)],
                                   indexing="ij"))
    ijk    = grid.reshape(3, -1).T
    inside = np.sum((ijk_to_xyz(ijk, affine) - center) ** 2, axis=1) <= radius_mm ** 2
    if inside.sum() == 0:
        return np.nan
    vals = vol[ijk[inside, 0], ijk[inside, 1], ijk[inside, 2]] > 0
    return float(vals.mean())



## 5. k-means binning (train only)

In [ ]:
# number of bins (2, 3 or 4)
# bin 0 = lowest mean ΔLEDD, bin K-1 = highest mean ΔLEDD (best bin)
KMEANS_K = 3

def fit_kmeans_train(df_group: pd.DataFrame) -> tuple:
    # k-means on train ΔLEDD only, bins relabelled so 0 = lowest and K-1 = highest
    # returns (train df with bins, best bin, k used)
    dtrain = df_group[df_group[COL_ID].isin(train_ids)].copy()
    if len(dtrain) < 2:
        return None, None

    k_use = min(KMEANS_K, len(dtrain))
    km = KMeans(n_clusters=k_use, random_state=RANDOM_STATE, n_init=KMEANS_N_INIT)
    raw_labels = km.fit_predict(dtrain[COL_DELTA].values.reshape(-1, 1))

    # sort centres ascending and remap labels to 0..K-1
    centre_order = np.argsort(km.cluster_centers_.ravel())
    remap = {old: new for new, old in enumerate(centre_order)}
    dtrain["KMeansBin"] = [remap[l] for l in raw_labels]

    best_bin = k_use - 1

    return dtrain, best_bin, k_use



## figure 1A. subject bin histograms (train + held-out test)

In [ ]:
def group_label(grp: str) -> str:
    target, side = grp.split("_")
    return f"{'Left' if side == 'L' else 'Right'} {'STN' if target == 'STN' else 'GPi'}"


fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

for ax, grp in zip(axes.ravel(), GROUPS):
    palette = STN_PALETTE if grp.startswith("STN") else GPI_PALETTE
    df_g = df[df["Group"] == grp]

    result = fit_kmeans_train(df_g)
    ax.set_title(group_label(grp), fontsize=12)
    ax.set_xlabel("ΔLEDD")
    ax.set_ylabel("Number of patients")

    if result[0] is None:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center",
                transform=ax.transAxes, color="grey")
        continue

    dtrain, best_bin, k_use = result
    bins = np.histogram_bin_edges(dtrain[COL_DELTA].values, bins="auto")

    for b in range(k_use):
        vals = dtrain.loc[dtrain["KMeansBin"] == b, COL_DELTA].values
        label = f"Train Bin {b}" + (" (best)" if b == best_bin else "")
        ax.hist(vals, bins=bins, alpha=ALPHA_HIST, color=palette[b],
                edgecolor="black", linewidth=0.4, label=label)

    dtest = df_g[df_g[COL_ID].isin(test_ids)]
    if not dtest.empty:
        ax.hist(dtest[COL_DELTA].values, bins=bins, histtype="step",
                linewidth=1.8, color="black", label="Test (held out)")

    ax.text(0.97, 0.97, f"Best bin: {best_bin}",
            transform=ax.transAxes, va="top", ha="right", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.6", alpha=0.85))
    ax.legend(fontsize=9, loc="upper left")

plt.suptitle("Aggregate Center, Subject Bins", fontsize=14, fontweight="bold")
fig.savefig(os.path.join(OUT_DIR, f"Fig1A_subject_bins.{FIG_EXT}"),
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print("Figure 1A saved.")

## 6-8. aggregate center computation and test-set metrics (requires NIfTI)

In [ ]:
# compute AC and test-set proximity metrics for each group
all_metrics = []
ac_records  = []
diag_rows   = []

for grp in GROUPS:
    hemi   = grp.split("_")[1]
    df_g   = df[df["Group"] == grp].copy()
    dtrain = df_g[df_g[COL_ID].isin(train_ids)].copy()
    dtest  = df_g[df_g[COL_ID].isin(test_ids)].copy()

    diag = {"Group": grp, "n_train": len(dtrain), "n_test": len(dtest)}

    # skip guards
    if len(dtrain) < MIN_TRAIN_GROUP:
        diag["status"] = f"SKIP: only {len(dtrain)} train rows"
        diag_rows.append(diag); continue
    if len(dtest) < MIN_TEST_REG:
        diag["status"] = f"SKIP: only {len(dtest)} test rows"
        diag_rows.append(diag); continue

    # k-means on train
    result = fit_kmeans_train(df_g)
    if result[0] is None:
        diag["status"] = "SKIP: insufficient train data for K-means"
        diag_rows.append(diag); continue
    dtrain, best_bin, k_use = result
    dtrain_best = dtrain[dtrain["KMeansBin"] == best_bin]

    bin_means = dtrain.groupby("KMeansBin")[COL_DELTA].mean()
    diag["best_bin"]   = best_bin
    diag["bin_means"]  = bin_means.to_dict()
    diag["n_best_bin"] = len(dtrain_best)

    # load train masks (best bin)
    imgs, vols, kept = [], [], []
    for pid in dtrain_best[COL_ID]:
        img = load_mask(pid, hemi)
        if img is None:
            continue
        vol = largest_cc((img.get_fdata() > 0.5).astype(np.uint8))
        imgs.append(img); vols.append(vol); kept.append(pid)

    diag["n_train_best_masks"] = len(kept)
    diag["missing_train_masks"] = len(dtrain_best) - len(kept)

    if len(kept) < MIN_TRAIN_BEST_BIN:
        diag["status"] = f"SKIP: only {len(kept)} masks in best bin"
        diag_rows.append(diag); continue

    # grid check and AC
    try:
        assert_same_grid(imgs)
    except ValueError as e:
        diag["status"] = f"SKIP: grid mismatch, {e}"
        diag_rows.append(diag); continue

    affine     = imgs[0].affine
    center_xyz = compute_ac_xyz(vols, affine)
    diag["AC_xyz"] = center_xyz.tolist()

    ac_records.append({
        "Group": grp,
        "Target": grp.split("_")[0],
        "Hemisphere": hemi,
        "n_train_best": len(kept),
        "AC_x": center_xyz[0],
        "AC_y": center_xyz[1],
        "AC_z": center_xyz[2],
    })

    # test-set metrics
    n_missing_test = 0
    for _, row in dtest.iterrows():
        img = load_mask(row[COL_ID], hemi)
        if img is None:
            n_missing_test += 1
            continue
        # grid consistency check
        if img.shape != imgs[0].shape or not np.allclose(img.affine, affine, atol=1e-4):
            print(f"  test grid mismatch for {row[COL_ID]} ({grp}), skipping")
            n_missing_test += 1
            continue

        vol = largest_cc((img.get_fdata() > 0.5).astype(np.uint8))
        all_metrics.append({
            COL_ID:        row[COL_ID],
            "Group":       grp,
            "Target":      grp.split("_")[0],
            "Hemisphere":  hemi,
            "Distance_mm": nearest_voxel_distance(vol, affine, center_xyz),
            "SphereOverlap": sphere_overlap(vol, affine, center_xyz),
            COL_DELTA:     float(row[COL_DELTA]),
            "Age":         float(row[COL_AGE]),
            "Sex":         int(row["Sex_binary"]),
        })

    diag["missing_test_masks"] = n_missing_test
    diag["n_test_with_metrics"] = len(dtest) - n_missing_test
    diag["status"] = "OK"
    diag_rows.append(diag)

df_metrics = pd.DataFrame(all_metrics)
df_ac      = pd.DataFrame(ac_records)
df_diag    = pd.DataFrame(diag_rows)

print("\nAggregate centers")
display(df_ac)

print("\nDiagnostics")
diag_cols = ["Group","status","n_train","n_test","best_bin",
             "n_best_bin","n_train_best_masks","n_test_with_metrics"]
display(df_diag.reindex(columns=diag_cols))

print(f"\nTotal test-set metric rows: {len(df_metrics)}")

## 9. statistical analysis, robust regression + permutation test

In [ ]:
def run_rlm(df_sub: pd.DataFrame, xcol: str):
    X = sm.add_constant(df_sub[[xcol, "Age", "Sex"]])
    return sm.RLM(df_sub[COL_DELTA], X, M=sm.robust.norms.HuberT()).fit()


def permutation_p(df_sub: pd.DataFrame, xcol: str,
                  n_perm: int = N_PERMUTATIONS, seed: int = RANDOM_STATE) -> float:
    rng  = np.random.default_rng(seed)
    obs  = abs(run_rlm(df_sub, xcol).params[xcol])
    X    = sm.add_constant(df_sub[[xcol, "Age", "Sex"]])
    y    = df_sub[COL_DELTA].values
    hits = sum(
        abs(sm.RLM(rng.permutation(y), X, M=sm.robust.norms.HuberT()).fit().params[xcol]) >= obs
        for _ in range(n_perm)
    )
    return (hits + 1) / (n_perm + 1)


stats_rows = []

for grp in GROUPS:
    d = df_metrics[df_metrics["Group"] == grp].copy()
    if len(d) < MIN_TEST_REG:
        print(f"  {grp}: skipped (n={len(d)})")
        continue

    for metric in ["Distance_mm", "SphereOverlap"]:
        d2 = d.dropna(subset=[metric, COL_DELTA, "Age", "Sex"])
        if len(d2) < MIN_TEST_REG:
            continue
        res   = run_rlm(d2, metric)
        p_raw = permutation_p(d2, metric)
        stats_rows.append({
            "Group":   grp,
            "Metric":  metric,
            "n_test":  len(d2),
            "beta":    round(res.params[metric], 4),
            "se":      round(res.bse[metric], 4),
            "p_perm":  round(p_raw, 4),
        })

df_stats = pd.DataFrame(stats_rows)

if not df_stats.empty:
    df_stats["sig"] = df_stats["p_perm"] < 0.05
    display(df_stats.sort_values("p_perm"))
else:
    print("No groups met minimum n for regression.")

## figure 2. distance to AC by k-means bin (test set)

In [ ]:
# one violin per bin per group
TARGET_COLORS = {
    "STN_L": ["#f8bbd0", "#f06292", "#ad1457", "#6a1b9a"],
    "STN_R": ["#f8bbd0", "#f06292", "#ad1457", "#6a1b9a"],
    "GPI_L": ["#ffe0b2", "#ffb74d", "#f57c00", "#e65100"],
    "GPI_R": ["#ffe0b2", "#ffb74d", "#f57c00", "#e65100"],
}

available_groups = [g for g in GROUPS if g in df_metrics["Group"].values]

if not available_groups:
    print("No metric data available, run the AC loop first.")
else:
    # assign test patients to bins using the train k-means centres
    df_metrics_binned = df_metrics.copy()

    for grp in available_groups:
        df_g = df[df["Group"] == grp]
        result = fit_kmeans_train(df_g)
        if result[0] is None:
            continue
        dtrain, best_bin, k_use = result

        k_use2 = min(KMEANS_K, len(dtrain))
        km2 = KMeans(n_clusters=k_use2, random_state=RANDOM_STATE, n_init=KMEANS_N_INIT)
        km2.fit(dtrain[COL_DELTA].values.reshape(-1, 1))

        # sort centres ascending to match bin numbering
        centre_order = np.argsort(km2.cluster_centers_.ravel())
        remap = {old: new for new, old in enumerate(centre_order)}

        mask = df_metrics_binned["Group"] == grp
        vals = df_metrics_binned.loc[mask, COL_DELTA].values.reshape(-1, 1)
        raw  = km2.predict(vals)
        df_metrics_binned.loc[mask, "KMeansBin"] = [remap[l] for l in raw]

    df_metrics_binned["KMeansBin"] = df_metrics_binned["KMeansBin"].astype("Int64")

    n_cols = len(available_groups)
    fig, axes = plt.subplots(1, n_cols,
                             figsize=(3.8 * n_cols, 5),
                             constrained_layout=True)
    if n_cols == 1:
        axes = [axes]

    for ax, grp in zip(axes, available_groups):
        palette = TARGET_COLORS[grp]
        d = df_metrics_binned[df_metrics_binned["Group"] == grp].dropna(
            subset=["Distance_mm", "KMeansBin"])

        if d.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes)
            continue

        bins_present = sorted(d["KMeansBin"].dropna().unique())
        bin_labels   = [f"Bin {int(b)}" + (" ★" if int(b) == KMEANS_K - 1 else "")
                        for b in bins_present]
        bin_data     = [d.loc[d["KMeansBin"] == b, "Distance_mm"].dropna().values
                        for b in bins_present]

        # violin
        parts = ax.violinplot(bin_data, positions=range(len(bins_present)),
                              showmedians=True, showextrema=True, widths=0.7)

        for j, pc in enumerate(parts["bodies"]):
            pc.set_facecolor(palette[int(bins_present[j])])
            pc.set_edgecolor("black")
            pc.set_alpha(0.75)
        for key in ["cmedians", "cmins", "cmaxes", "cbars"]:
            parts[key].set_color("black")
            parts[key].set_linewidth(1.2)

        # jittered points
        rng = np.random.default_rng(42)
        for j, (bdata, b) in enumerate(zip(bin_data, bins_present)):
            jitter = rng.uniform(-0.12, 0.12, size=len(bdata))
            ax.scatter(j + jitter, bdata,
                       color=palette[int(b)], edgecolors="black",
                       linewidths=0.5, s=22, alpha=0.8, zorder=3)

        # shade best bin
        best_pos = bins_present.index(KMEANS_K - 1) if (KMEANS_K - 1) in bins_present else None
        if best_pos is not None:
            ax.axvspan(best_pos - 0.45, best_pos + 0.45,
                       color="gold", alpha=0.15, zorder=0, label="Best bin")

        # Welch t-test of each bin vs best bin
        best_data = bin_data[bins_present.index(KMEANS_K - 1)] if (KMEANS_K - 1) in bins_present else None
        if best_data is not None and len(best_data) >= 3:
            y_max = max(v.max() for v in bin_data if len(v) > 0)
            step  = (ax.get_ylim()[1] - y_max) * 0.05 if ax.get_ylim()[1] > y_max else 3
            for j, (bdata, b) in enumerate(zip(bin_data, bins_present)):
                if int(b) == KMEANS_K - 1 or len(bdata) < 3:
                    continue
                _, p = stats.ttest_ind(best_data, bdata, equal_var=False)
                sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
                xpos = (j + best_pos) / 2
                ax.annotate("", xy=(j, y_max + step), xytext=(best_pos, y_max + step),
                            arrowprops=dict(arrowstyle="-", color="black", lw=1))
                ax.text(xpos, y_max + step * 1.3, sig, ha="center", fontsize=9)

        ax.set_xticks(range(len(bins_present)))
        ax.set_xticklabels(bin_labels, fontsize=9)
        ax.set_title(group_label(grp), fontsize=11, fontweight="bold")
        ax.set_ylabel("Distance to aggregate center (mm)" if grp == available_groups[0] else "",
                      fontsize=10)
        ax.set_xlabel("ΔLEDD bin", fontsize=10)

        # n per bin
        for j, bdata in enumerate(bin_data):
            ax.text(j, ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.06,
                    f"n={len(bdata)}", ha="center", fontsize=8, color="0.4")

    plt.suptitle(
        "Distance to aggregate center by outcome bin (test set)\n"
        "★ = best bin  |  gold shading = best bin  |  brackets = Welch t-test vs best",
        fontsize=11, fontweight="bold"
    )
    fig.savefig(os.path.join(OUT_DIR, f"Fig2_violin_distance_by_bin.{FIG_EXT}"),
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print("Figure 2 saved.")

## figure 2B. proximity vs ΔLEDD

In [ ]:
if not df_metrics.empty:
    fig, axes = plt.subplots(2, len(available_groups),
                             figsize=(4 * len(available_groups), 7),
                             constrained_layout=True)
    if len(available_groups) == 1:
        axes = axes[:, np.newaxis]

    metrics_info = [
        ("SphereOverlap", "Fraction of AC sphere\noverlapping VTA"),
        ("Distance_mm",   "Distance to AC (mm)"),
    ]

    for col_i, grp in enumerate(available_groups):
        d      = df_metrics[df_metrics["Group"] == grp]
        colors = TARGET_COLORS.get(grp, ["grey", "grey", "black", "black"])
        col_lo = colors[0]
        col_hi = colors[-1]

        for row_i, (metric, xlabel) in enumerate(metrics_info):
            ax = axes[row_i, col_i]
            d2 = d.dropna(subset=[metric, COL_DELTA])
            if len(d2) < MIN_TEST_REG:
                ax.text(0.5, 0.5, f"n={len(d2)}\n(insufficient)",
                        ha="center", va="center",
                        transform=ax.transAxes, color="grey")
                if row_i == 0:
                    ax.set_title(group_label(grp), fontsize=11, fontweight="bold")
                continue

            sns.regplot(data=d2, x=metric, y=COL_DELTA, ax=ax,
                        scatter_kws=dict(s=30, alpha=0.7, color=col_hi),
                        line_kws=dict(lw=2, color=col_lo), ci=95, color=col_hi)

            # annotate p_perm
            if not df_stats.empty:
                stat_row = df_stats[
                    (df_stats["Group"] == grp) & (df_stats["Metric"] == metric)
                ]
                if not stat_row.empty:
                    sr  = stat_row.iloc[0]
                    sig = " *" if sr["p_perm"] < 0.05 else ""
                    ann = f"β={sr['beta']:.3f}\np_perm={sr['p_perm']:.3f}{sig}"
                    ax.text(0.97, 0.97, ann, transform=ax.transAxes,
                            ha="right", va="top", fontsize=8,
                            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.7"))

            if row_i == 0:
                ax.set_title(group_label(grp), fontsize=11, fontweight="bold")
            ax.set_xlabel(xlabel, fontsize=9)
            ax.set_ylabel("ΔLEDD" if col_i == 0 else "", fontsize=10)

    plt.suptitle(
        "AC proximity vs medication reduction\n(robust regression, held-out test set)",
        fontsize=12, fontweight="bold"
    )
    fig.savefig(os.path.join(OUT_DIR, f"Fig2B_regression_scatter.{FIG_EXT}"),
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print("Figure 2B saved.")

## figure 3. UCSF aggregate centers vs published sweet spots (3D)

published STN sweet-spot coordinates for PD, all from probabilistic VTA mapping in Lead-DBS, MNI space (mm). right hemisphere as published, left mirrored.

| study | outcome | n | key finding |
|---|---|---|---|
| Dembek et al. 2019 *Ann Neurol* | overall UPDRS-III improvement | 21 (acute) + 63 (chronic) | dorsolateral STN border; R sweet spot at (12.5, −12.7, −5.4) |
| Akram et al. 2017 *Neuroimage* | combined rigidity/bradykinesia/tremor | 20 | supero-lateral STN/ZI boundary; R peak ≈ (10, −13, −7) |
| Tödt et al. 2022 *Mov Disord* (EARLYSTIM) | UPDRS-III at 24 months | 69 | R sweet spot (11.6, −13.1, −6.3) |
| Hacker et al. 2023 *Ann Neurol* | motor progression and symptomatic improvement | 14 (early PD) | dorsolateral STN; R peaks ~(11.2–11.25, −13.6–13.7, −7.4) |

In [ ]:
# published STN sweet spots, MNI152NLin2009bAsym (mm)
#   Dembek et al. 2019  Ann Neurol 86:527-538
#   Akram et al.  2017  Neuroimage 158:332-345
#   Tödt et al.   2022  Mov Disord 37:291-301
#   Hacker et al. 2023  Ann Neurol 94:320-334
LITERATURE_SPOTS = pd.DataFrame([

    # Dembek 2019, overall motor improvement
    dict(Label="Dembek 2019\n(R)", Target="STN", Side="R",
         x= 12.50, y=-12.72, z=-5.38, Source="Dembek et al. 2019"),
    dict(Label="Dembek 2019\n(L)", Target="STN", Side="L",
         x=-12.50, y=-12.72, z=-5.38, Source="Dembek et al. 2019"),

    # Akram 2017, maximum overall efficacy
    dict(Label="Akram 2017\n(R)", Target="STN", Side="R",
         x= 10.0, y=-13.0, z=-7.0, Source="Akram et al. 2017"),
    dict(Label="Akram 2017\n(L)", Target="STN", Side="L",
         x=-10.0, y=-13.0, z=-7.0, Source="Akram et al. 2017"),

    # Tödt 2022 (EARLYSTIM), UPDRS-III
    dict(Label="Tödt 2022\nEARLYSTIM (R)", Target="STN", Side="R",
         x= 11.6, y=-13.1, z=-6.3, Source="Tödt et al. 2022"),
    dict(Label="Tödt 2022\nEARLYSTIM (L)", Target="STN", Side="L",
         x=-11.6, y=-13.1, z=-6.3, Source="Tödt et al. 2022"),

    # Hacker 2023, slower motor progression
    dict(Label="Hacker 2023\nMotor Prog (R)", Target="STN", Side="R",
         x= 11.25, y=-13.56, z=-7.44, Source="Hacker et al. 2023"),
    dict(Label="Hacker 2023\nMotor Prog (L)", Target="STN", Side="L",
         x=-11.25, y=-13.56, z=-7.44, Source="Hacker et al. 2023"),

    # Hacker 2023, symptomatic improvement
    dict(Label="Hacker 2023\nSympt (R)", Target="STN", Side="R",
         x= 11.2, y=-13.7, z=-7.4, Source="Hacker et al. 2023"),
    dict(Label="Hacker 2023\nSympt (L)", Target="STN", Side="L",
         x=-11.2, y=-13.7, z=-7.4, Source="Hacker et al. 2023"),
])

# add UCSF STN aggregate centers (no GPi for this comparison)
ucsf_stn_rows = []
for _, r in df_ac.iterrows():
    if r["Target"] == "STN":
        ucsf_stn_rows.append(dict(
            Label=f"UCSF {r['Group']}",
            Target="STN",
            Side=r["Hemisphere"],
            x=r["AC_x"], y=r["AC_y"], z=r["AC_z"],
            Source="UCSF (this study)",
        ))
df_ucsf_stn = pd.DataFrame(ucsf_stn_rows)
df_all_spots = pd.concat([LITERATURE_SPOTS, df_ucsf_stn], ignore_index=True)

# marker per source
SOURCE_MARKERS = {
    "Dembek et al. 2019":  ("s",  9),
    "Akram et al. 2017":   ("^",  9),
    "Tödt et al. 2022":    ("D",  9),
    "Hacker et al. 2023":  ("P",  9),
    "UCSF (this study)":   ("*", 14),
}

# colour by hemisphere
COLOR_BY = {
    "L": "#c62828",
    "R": "#f48fb1",
}

# 3D scatter
fig = plt.figure(figsize=(11, 7))
ax3d = fig.add_subplot(111, projection="3d")

legend_handles = []
seen_sources   = set()

for _, row in df_all_spots.iterrows():
    color          = COLOR_BY.get(row["Side"], "grey")
    marker, msize  = SOURCE_MARKERS.get(row["Source"], ("o", 8))
    is_ucsf        = row["Source"] == "UCSF (this study)"
    edgew          = 2.0 if is_ucsf else 0.5
    zorder         = 5   if is_ucsf else 3

    ax3d.scatter(row["x"], row["y"], row["z"],
                 c=color, marker=marker, s=msize**2,
                 edgecolors="black", linewidths=edgew,
                 zorder=zorder, alpha=0.9)

    if row["Source"] not in seen_sources:
        seen_sources.add(row["Source"])
        m, ms = SOURCE_MARKERS.get(row["Source"], ("o", 8))
        legend_handles.append(
            plt.Line2D([0], [0], marker=m, color="w",
                       markerfacecolor="grey", markeredgecolor="black",
                       markersize=ms * 0.9, label=row["Source"], linestyle="None")
        )

# hemisphere colour patches
for side, col in COLOR_BY.items():
    legend_handles.append(
        mpatches.Patch(facecolor=col,
                       label=f"STN {'Left' if side == 'L' else 'Right'}")
    )

ax3d.set_xlabel("x (mm)", fontsize=10, labelpad=8)
ax3d.set_ylabel("y (mm)", fontsize=10, labelpad=8)
ax3d.set_zlabel("z (mm)", fontsize=10, labelpad=8)
ax3d.set_title("UCSF STN Aggregate Centers vs Published PD Sweet Spots\n(MNI space)",
               fontsize=12)
ax3d.legend(handles=legend_handles, fontsize=8, loc="upper left",
            bbox_to_anchor=(1.02, 1.0), frameon=True)
ax3d.view_init(elev=20, azim=-60)

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f"Fig3_3D_sweetspots.{FIG_EXT}"),
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

## figure 4. distances between sweet spots

In [ ]:
# lower-left triangle = left hemisphere, upper-right = right hemisphere
df_left  = df_all_spots[df_all_spots["Side"] == "L"].reset_index(drop=True)
df_right = df_all_spots[df_all_spots["Side"] == "R"].reset_index(drop=True)
assert list(df_left["Source"]) == list(df_right["Source"]), "Source order mismatch"

def clean_label(row):
    src = row["Source"].replace("UCSF (this study)", "UCSF").replace(" et al.", "")
    lbl = row["Label"]
    if "Motor Prog" in lbl: return src + "\n(motor prog.)"
    elif "Sympt" in lbl:    return src + "\n(symptomatic)"
    return src

labels   = [clean_label(row) for _, row in df_left.iterrows()]
n        = len(labels)
coords_L = df_left[["x", "y", "z"]].values
coords_R = df_right[["x", "y", "z"]].values
dmat_L   = cdist(coords_L, coords_L, metric="euclidean")
dmat_R   = cdist(coords_R, coords_R, metric="euclidean")

combined = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i > j:   combined[i, j] = dmat_L[i, j]
        elif i < j: combined[i, j] = dmat_R[i, j]

display_mat = combined.astype(float)
np.fill_diagonal(display_mat, np.nan)
vmax = np.nanmax(display_mat)

fig, ax = plt.subplots(figsize=(9, 7.5))
fig.subplots_adjust(left=0.22, right=0.82, top=0.85, bottom=0.22)

im = ax.imshow(display_mat, cmap="YlOrRd_r", aspect="equal",
               vmin=0, vmax=vmax, interpolation="nearest")

# white diagonal tiles
for i in range(n):
    ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1,
                                facecolor="white", edgecolor="0.8", lw=0.5, zorder=2))

# cell values
for i in range(n):
    for j in range(n):
        if i == j: continue
        val = combined[i, j]
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8,
                color="white" if val < vmax * 0.35 else "black", zorder=3)

# axis labels
ax.set_xticks(range(n))
ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8.5)
ax.set_yticks(range(n))
ax.set_yticklabels(labels, fontsize=8.5)

# dashed diagonal divider
ax.plot([-0.5, n-0.5], [-0.5, n-0.5], color="0.4", lw=1.5, linestyle="--", zorder=4)

mid = (n - 1) / 2

# hemisphere labels, right along the top edge
ax.annotate("", xy=(n-0.5, -0.5), xytext=(0.5, -0.5),
            annotation_clip=False,
            arrowprops=dict(arrowstyle="-[, widthB=0.0", color="#c0392b", lw=2.0))
ax.text(mid + 0.25, -1.55, "Right hemisphere", ha="center", va="top",
        fontsize=10, fontweight="bold", color="#c0392b",
        transform=ax.transData, clip_on=False)

# left along the left edge
ax.annotate("", xy=(-0.5, n-0.5), xytext=(-0.5, 0.5),
            annotation_clip=False,
            arrowprops=dict(arrowstyle="-[, widthB=0.0", color="#1565c0", lw=2.0))
ax.text(-1.65, mid + 0.25, "Left hemisphere", ha="right", va="center",
        fontsize=10, fontweight="bold", color="#1565c0",
        rotation=90, transform=ax.transData, clip_on=False)

# UCSF row/col highlight
ucsf_idx = next(i for i, r in df_left.iterrows() if r["Source"] == "UCSF (this study)")
if ucsf_idx > 0:
    ax.add_patch(plt.Rectangle((ucsf_idx-0.5, -0.5), 1, ucsf_idx,
                                fill=False, edgecolor="#e91e63", lw=2.5,
                                clip_on=False, zorder=5))
if ucsf_idx < n - 1:
    ax.add_patch(plt.Rectangle((-0.5, ucsf_idx-0.5), ucsf_idx, 1,
                                fill=False, edgecolor="#e91e63", lw=2.5,
                                clip_on=False, zorder=5))

# minor grid
ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
ax.grid(which="minor", color="0.75", linewidth=0.5, zorder=1)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cbar.set_label("Euclidean distance (mm)", fontsize=10)

ax.set_title(
    "Pairwise Euclidean Distances Between STN Sweet Spots (mm)\n"
    "Pink borders = UCSF aggregate centers",
    fontsize=11, fontweight="bold", pad=14
)

fig.savefig(os.path.join(OUT_DIR, f"Fig4_distance_heatmap.{FIG_EXT}"),
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print("Figure 4 saved.")

## summary tables

In [ ]:
# table 1, aggregate center coordinates
print("=== Table 1: Aggregate center coordinates (MNI mm) ===")
display(df_ac.rename(columns={
    "Group": "Group", "Target": "Target", "Hemisphere": "Hemi",
    "n_train_best": "N (train best-bin)", "AC_x": "x", "AC_y": "y", "AC_z": "z"
}))

# table 2, regression results
print("\n=== Table 2: Robust regression results ===")
if not df_stats.empty:
    display(df_stats[[
        "Group", "Metric", "n_test", "beta", "se", "p_perm"
    ]].sort_values(["Group", "Metric"]))
else:
    print("No regression results, check minimum-n thresholds.")

# table 3, UCSF vs literature distances
print("\n=== Table 3: UCSF vs literature, Euclidean distances (mm) ===")
dist_mat     = cdist(df_all_spots[["x", "y", "z"]].values, df_all_spots[["x", "y", "z"]].values)
short_labels = df_all_spots["Label"].str.replace("\n", " ")
ucsf_idx = df_all_spots[df_all_spots["Source"] == "UCSF (this study)"].index.tolist()
lit_idx  = df_all_spots[df_all_spots["Source"] != "UCSF (this study)"].index.tolist()

if ucsf_idx and lit_idx:
    sub_dist = pd.DataFrame(
        dist_mat[np.ix_(ucsf_idx, lit_idx)],
        index=short_labels.iloc[ucsf_idx],
        columns=short_labels.iloc[lit_idx],
    ).round(2)
    display(sub_dist)

## save outputs

In [ ]:
df_ac.to_csv(os.path.join(OUT_DIR, "aggregate_centers.csv"), index=False)
df_metrics.to_csv(os.path.join(OUT_DIR, "test_metrics.csv"), index=False)
if not df_stats.empty:
    df_stats.to_csv(os.path.join(OUT_DIR, "regression_results.csv"), index=False)
df_all_spots.to_csv(os.path.join(OUT_DIR, "all_sweet_spots.csv"), index=False)

print("Saved to:", OUT_DIR)

In [ ]:
# all AC results in one CSV
frames = []

# proximity regression results
if not df_stats.empty:
    d = df_stats.copy()
    d.insert(0, 'Analysis', 'AC_Proximity_Regression')
    d.insert(1, 'Sample', 'test_set')
    frames.append(d)

# per-patient test-set proximity metrics
if not df_metrics.empty:
    d = df_metrics.copy()
    d.insert(0, 'Analysis', 'AC_Patient_Metrics')
    d.insert(1, 'Sample', 'test_set')
    frames.append(d)

# aggregate center coordinates
if not df_ac.empty:
    d = df_ac.copy()
    d.insert(0, 'Analysis', 'Aggregate_Centers')
    d.insert(1, 'Sample', 'train_best_bin')
    frames.append(d)

if frames:
    out = pd.concat(frames, ignore_index=True, sort=False)
    out_path = os.path.join(OUT_DIR, 'AC_results_all.csv')
    out.drop(columns=[c for c in out.columns if c.startswith('_')], errors='ignore').to_csv(out_path, index=False)
    print(f'saved {len(out)} rows to {out_path}')
else:
    print("No result dataframes found to export.")